In [ ]:
!pip install cirq

In [ ]:
import cirq
import sympy

num_features = 10

# create qubits
qubits = cirq.LineQubit.range(num_features)

# parameters
x = sympy.symbols('x0:10')

circuit = cirq.Circuit()

# rotations
for i, q in enumerate(qubits):
    circuit.append(cirq.rx(x[i])(q))
    circuit.append(cirq.ry(2*x[i])(q))
    circuit.append(cirq.rz(0.5*x[i])(q))

# entanglement
for i in range(num_features - 1):
    circuit.append(cirq.CNOT(qubits[i], qubits[i+1]))

print(circuit)

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes[:]
dff

In [ ]:
dff.info()

**<h1>Qsvm**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.svm import SVC
import cirq
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
N_RUNS = 1
TEST_SIZE = 0.8
NUM_REPEATS = 65       # repeated random subsampling
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "cirq_qsvm_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

# Fixed feature sequence (0-based indexing)
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# LOAD DATA
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset for demonstration...")
    np.random.seed(42)
    n_samples = 200
    n_features = 15
    X_pos = np.random.randn(n_samples // 2, n_features) + 1
    X_neg = np.random.randn(n_samples // 2, n_features) - 1
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    columns = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X, columns=columns)
    dfs_smotes[TARGET] = y
    print(f"Sample dataset created: {dfs_smotes.shape}")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print(f"Using fixed features: {feature_names}")

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ============================================================
# CIRCUIT ENCODING
# ============================================================
def dense_angle_encoding_cirq(x, qubits, circuit_depth=1, n_reuploads=1):
    """Dense angle encoding with multiple depth & reuploads"""
    circuit = cirq.Circuit()
    for _ in range(circuit_depth):
        for _ in range(n_reuploads):
            for i, q in enumerate(qubits):
                val = x[i] if i < len(x) else 0.0
                circuit.append(cirq.rx(val)(q))
                circuit.append(cirq.ry(2.0 * val)(q))
                circuit.append(cirq.rz(0.5 * val)(q))
            for i in range(len(qubits) - 1):
                circuit.append(cirq.CNOT(qubits[i], qubits[i + 1]))
    return circuit

def compute_quantum_kernel(X1, X2, circuit_depth=1, n_reuploads=1):
    """Compute Cirq QSVM kernel matrix"""
    n_qubits = X1.shape[1]
    qubits = [cirq.LineQubit(i) for i in range(n_qubits)]
    simulator = cirq.Simulator()

    def state_vector(x):
        circuit = dense_angle_encoding_cirq(x, qubits, circuit_depth, n_reuploads)
        result = simulator.simulate(circuit)
        return result.final_state_vector

    K = np.zeros((X1.shape[0], X2.shape[0]))
    for i, x1 in enumerate(X1):
        psi1 = state_vector(x1)
        for j, x2 in enumerate(X2):
            psi2 = state_vector(x2)
            K[i, j] = np.abs(np.vdot(psi1, psi2))**2
    return K

# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING
# ============================================================
summary = []
total_jobs = NUM_REPEATS
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS + 1):
    # Train-Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED_BASE * split
    )

    # Imputation + Scaling
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    # ---------- QSVM / Quantum Kernel SVM ----------
    start = time.time()
    K_train = compute_quantum_kernel(X_train, X_train)
    K_test = compute_quantum_kernel(X_test, X_train)

    clf = SVC(kernel="precomputed", probability=True)
    clf.fit(K_train, y_train)
    y_prob = clf.predict_proba(K_test)[:, 1]
    runtime = time.time() - start

    # ---------- Metrics ----------
    metrics = compute_metrics(y_test, y_prob)
    completed_jobs += 1
    progress = (completed_jobs / total_jobs) * 100

    print(
        f"[{progress:6.2f}%] "
        f"Cirq_QSVM | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy": metrics["Accuracy"],
        "ROC_AUC": metrics["ROC-AUC"],
        "F1": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }
    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("\n===== FINAL SORTED RESULTS =====")
print(summary_df)
print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")
print("\nDONE.")

**<h1>Qknn**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
import cirq
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
N_RUNS = 1
TEST_SIZE = 0.8
NUM_REPEATS = 65
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "cirq_qknn_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

# QKNN parameter
K_NEIGHBORS = 3

# Fixed feature sequence (0-based indexing)
FIXED_FEATURES_IDX = [8, 10, 9, 13, 11, 14, 5, 3, 6, 2]

# ============================================================
# LOAD DATA
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Creating synthetic dataset for demonstration...")
    np.random.seed(42)
    n_samples = 200
    n_features = 15
    X_pos = np.random.randn(n_samples // 2, n_features) + 1
    X_neg = np.random.randn(n_samples // 2, n_features) - 1
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n_samples // 2) + [0] * (n_samples // 2))
    columns = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X, columns=columns)
    dfs_smotes[TARGET] = y
    print(f"Sample dataset created: {dfs_smotes.shape}")

X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print(f"Using fixed features: {feature_names}")

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0,
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0,
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# ============================================================
# CIRCUIT ENCODING
# ============================================================
def dense_angle_encoding_cirq(x, qubits, circuit_depth=1, n_reuploads=1):
    circuit = cirq.Circuit()
    for _ in range(circuit_depth):
        for _ in range(n_reuploads):
            for i, q in enumerate(qubits):
                val = x[i] if i < len(x) else 0.0
                circuit.append(cirq.rx(val)(q))
                circuit.append(cirq.ry(2.0 * val)(q))
                circuit.append(cirq.rz(0.5 * val)(q))
            for i in range(len(qubits) - 1):
                circuit.append(cirq.CNOT(qubits[i], qubits[i + 1]))
    return circuit

def compute_quantum_kernel(X1, X2, circuit_depth=1, n_reuploads=1):
    n_qubits = X1.shape[1]
    qubits = [cirq.LineQubit(i) for i in range(n_qubits)]
    simulator = cirq.Simulator()

    def state_vector(x):
        circuit = dense_angle_encoding_cirq(x, qubits, circuit_depth, n_reuploads)
        result = simulator.simulate(circuit)
        return result.final_state_vector

    K = np.zeros((X1.shape[0], X2.shape[0]))
    for i, x1 in enumerate(X1):
        psi1 = state_vector(x1)
        for j, x2 in enumerate(X2):
            psi2 = state_vector(x2)
            K[i, j] = np.abs(np.vdot(psi1, psi2))**2
    return K

# ============================================================
# QKNN IMPLEMENTATION
# ============================================================
def qknn_predict(K_test, y_train, k=3):
    """
    Predict probabilities using quantum kernel similarity.
    """
    probs = []

    for i in range(K_test.shape[0]):

        # similarity between test sample and all train samples
        sims = K_test[i]

        # top-k indices
        idx = np.argsort(sims)[-k:]

        # neighbor labels
        labels = y_train[idx]

        # probability = mean of labels
        prob = np.mean(labels)

        probs.append(prob)

    return np.array(probs)

# ============================================================
# MAIN LOOP WITH REPEATED RANDOM SUBSAMPLING
# ============================================================
summary = []
total_jobs = NUM_REPEATS
completed_jobs = 0

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS + 1):

    # Train-Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y, test_size=TEST_SIZE, stratify=y,
        random_state=RANDOM_SEED_BASE * split
    )

    # Imputation + Scaling
    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()

    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    # ---------- QKNN ----------
    start = time.time()

    K_train = compute_quantum_kernel(X_train, X_train)
    K_test = compute_quantum_kernel(X_test, X_train)

    y_prob = qknn_predict(K_test, y_train, K_NEIGHBORS)

    runtime = time.time() - start

    # ---------- Metrics ----------
    metrics = compute_metrics(y_test, y_prob)

    completed_jobs += 1
    progress = (completed_jobs / total_jobs) * 100

    print(
        f"[{progress:6.2f}%] "
        f"Cirq_QKNN | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split": split,
        "Accuracy": metrics["Accuracy"],
        "ROC_AUC": metrics["ROC-AUC"],
        "F1": metrics["F1"],
        "Precision": metrics["Precision"],
        "Sensitivity": metrics["Sensitivity"],
        "Specificity": metrics["Specificity"],
        "Kappa": metrics["Kappa"],
        "Runtime_sec": runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH, index=False)

# ============================================================
# FINAL SUMMARY
# ============================================================
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("\n===== FINAL SORTED RESULTS =====")
print(summary_df)

print(f"\nAuto-saved to: {AUTO_SAVE_PATH}")

print("\nDONE.")

**<h1>Qboost**

In [ ]:
import numpy as np
import pandas as pd
import time
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
import cirq
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
TEST_SIZE = 0.8
NUM_REPEATS = 65
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "cirq_qboost_fixed_features_results.csv"
TARGET = "LUNG_CANCER"

FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

# ============================================================
# LOAD DATA
# ============================================================
X_raw = dfs_smotes.iloc[:, FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true, y_prob, thr=0.5):

    y_pred = (y_prob >= thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Sensitivity": tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity": tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa": cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# QUANTUM WEAK LEARNER
# ============================================================
class QuantumWeakLearner:

    def __init__(self, n_qubits):
        self.n_qubits = n_qubits
        self.weights = np.random.uniform(0,2*np.pi,n_qubits)

    def circuit(self, x):

        qubits = cirq.LineQubit.range(self.n_qubits)
        circuit = cirq.Circuit()

        for i,q in enumerate(qubits):
            circuit.append(cirq.rx(x[i])(q))
            circuit.append(cirq.ry(2*x[i])(q))
            circuit.append(cirq.rz(0.5*x[i])(q))

        for i in range(self.n_qubits-1):
            circuit.append(cirq.CNOT(qubits[i],qubits[i+1]))

        for i,q in enumerate(qubits):
            circuit.append(cirq.ry(self.weights[i])(q))

        return circuit

    def predict(self,X):

        simulator = cirq.Simulator()
        preds=[]

        for x in X:

            circuit=self.circuit(x)
            result=simulator.simulate(circuit)
            state=result.final_state_vector

            prob0=np.sum(np.abs(state[:len(state)//2])**2)

            preds.append(1 if prob0>0.5 else 0)

        return np.array(preds)

# ============================================================
# QBOOST
# ============================================================
class QBoost:

    def __init__(self,n_estimators=5):
        self.n_estimators=n_estimators
        self.learners=[]
        self.alphas=[]

    def fit(self,X,y):

        n_samples=X.shape[0]
        weights=np.ones(n_samples)/n_samples

        for _ in range(self.n_estimators):

            learner=QuantumWeakLearner(X.shape[1])

            y_pred=learner.predict(X)

            err=np.sum(weights*(y_pred!=y))/np.sum(weights)

            if err>0.5:
                continue

            alpha=0.5*np.log((1-err)/max(err,1e-10))

            weights*=np.exp(-alpha*(2*y-1)*(2*y_pred-1))
            weights/=np.sum(weights)

            self.learners.append(learner)
            self.alphas.append(alpha)

        return self

    def predict_proba(self,X):

        score=np.zeros(X.shape[0])

        for learner,alpha in zip(self.learners,self.alphas):

            pred=learner.predict(X)
            score+=alpha*(2*pred-1)

        prob=1/(1+np.exp(-score))

        return np.vstack([1-prob,prob]).T

# ============================================================
# TRAIN LOOP
# ============================================================
summary=[]
total_jobs = NUM_REPEATS
completed_jobs = 0

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test=train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    clf=QBoost(n_estimators=5)
    clf.fit(X_train,y_train)

    y_prob=clf.predict_proba(X_test)[:,1]

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    completed_jobs += 1
    progress = (completed_jobs/total_jobs)*100

    print(
        f"[{progress:6.2f}%] "
        f"Cirq_QBoost | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    summary.append({
        "Split":split,
        **metrics,
        "Runtime_sec":runtime
    })

df_summary=pd.DataFrame(summary)
print(df_summary)